In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [6]:
chain = prompt | llm | output_parser

In [7]:
# Run LLM on a given article
def run_llm(headline, body):
    return chain.invoke({"headline": headline, "body": body})

## NER Model

In [8]:
import spacy
from span_marker import SpanMarkerModel

In [9]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [10]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [11]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [12]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [13]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [14]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

## Pipeline Entry Point

In [15]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [16]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(20)
# raw_df = full_df
len(raw_df)


20

In [17]:
# raw_df = pd.read_csv(sample_data_path)

In [18]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
7311,0000017e-9213-d7bb-a97f-bff3f38a0001,Article,"WPI student found dead in apartment, marking t...","WPI student found dead in apartment, marking t...",Sam Turken,NaN,Education,NaN,/education/2022/01/25/wpi-student-found-dead-i...,Tue Jan 25 13:02:18 EST 2022,TRUE,Another Worcester Polytechnic Institute studen...
4153,00000179-d37f-d770-a57f-f77f3c9c0001,Article,The Home For Little Wanderers Sees Increased N...,The Home For Little Wanderers Sees Increased N...,Joe Mathieu,NaN,Local News,NaN,/local-news/2021/06/03/the-home-for-little-wan...,Thu Jun 03 16:47:27 EDT 2021,TRUE,<i>A year of shutdowns and remote learning has...
4674,0000017a-8257-dc5c-a77a-8ad76dfe0001,Article,The Ever Given Has Set Sail From The Suez Cana...,The Ever Given Has Set Sail From The Suez Cana...,The Associated Press,NaN,National News,NaN,/national-news/2021/07/08/the-ever-given-has-s...,Wed Jul 07 14:27:00 EDT 2021,TRUE,"ISMAILIA, Egypt — Egyptian authorities announc..."
6419,0000017d-1962-d269-a3fd-bf7b76d40001,Article,Republicans and Democrats contemplate a future...,Republicans and Democrats contemplate a future...,Ron Elving,NaN,National News,NaN,/national-news/2021/11/13/republicans-and-demo...,Sat Nov 13 07:00:00 EST 2021,TRUE,Let's assume you have spent at least a few min...
6379,0000017d-0c0e-d7af-a3ff-ec3e1c440001,Article,"""For us, Veterans Day is every day,” says one ...","""For us, Veterans Day is every day,” says one ...",Craig LeMoult,NaN,Local News,NaN,/local-news/2021/11/11/for-us-veterans-day-is-...,Thu Nov 11 07:46:35 EST 2021,TRUE,"Carlos De León, who grew up in Lawrence, serve..."
10694,00000183-f771-d28d-a9f3-f7f9d78f0001,Article,Sweeps of Mass. and Cass encampments return as...,Sweeps of Mass. and Cass encampments return as...,Tori Bedford,NaN,Local News,NaN,/local-news/2022/10/20/sweeps-of-mass-and-cass...,Thu Oct 20 18:53:03 EDT 2022,TRUE,Following a city-sanctioned sweep of a new enc...
10770,00000184-18e3-d508-a3bf-5eff52940003,Article,NPR reporting on Oregon theater death threats ...,NPR reporting on Oregon theater death threats ...,Chloe Veltman,NaN,National News,NaN,/national-news/2022/10/27/npr-reporting-on-ore...,Thu Oct 27 05:00:00 EDT 2022,TRUE,"When<a href=""https://www.osfashland.org/en/art..."
6351,0000017d-045b-df97-afff-3f5f040b0001,Article,A secret tape made after Columbine shows the N...,A secret tape made after Columbine shows the N...,Tim Mak,NaN,National News,NaN,/national-news/2021/11/09/a-secret-tape-made-a...,Tue Nov 09 05:00:00 EST 2021,TRUE,Soon after the Columbine High School shooting ...
5433,0000017b-989c-df60-a9ff-f99fd2b20001,Article,How To Avoid The Next Afghanistan: Follow The ...,How To Avoid The Next Afghanistan: Follow The ...,Harvey Silverglate,NaN,Commentary,NaN,/commentary/2021/08/31/how-to-avoid-the-next-a...,Tue Aug 31 06:00:52 EDT 2021,TRUE,World War II was the last sustained military e...
10484,00000183-a2e7-d687-afef-feff35e80001,Article,What's at stake for the Voting Rights Act in t...,What's at stake for the Voting Rights Act in t...,"Paris Alston, Jeremy Siegel, Daniel Medwed",NaN,Local News,NaN,/local-news/2022/10/04/whats-at-stake-for-the-...,Tue Oct 04 14:56:37 EDT 2022,TRUE,"<i>It&#39;s early October, which means a new t..."


The ML Model honestly just needs the `id`, `header`, and `body`.

In [19]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [20]:
# For Testing Purposes Only
# df = df[:20]

In [21]:
df["llama_prediction"] = None # Add the llama_prediction

Remove Duplicates (if any)

In [22]:
duplicates = df.duplicated(subset=['hl1'])

In [23]:
print(duplicates.value_counts())

False    20
Name: count, dtype: int64


In [24]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [25]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

100%|██████████| 20/20 [00:00<?, ?it/s]


Clean the Body and Header with Regex

In [26]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 20/20 [00:00<?, ?it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary

In [27]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

In [28]:
# TODO: check for unwanted locations
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        # if (key in lowercase_header):
        if (location.lower() in lowercase_header):
            return [location, known_title_locs[location]]   
    return None

In [29]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 20/20 [00:00<?, ?it/s]


In [30]:
df["Explicit_Pass"].value_counts().head(10)

Explicit_Pass
[New, {'lat': 40.9914896, 'lon': -72.47591899999999}]    1
Name: count, dtype: int64

### NER Code First Pass

In [31]:
# TODO: Can optimize further by from the get go checking that they aren't unwanted and return instead after finding a facility or organization if no facility is found 
# Run NER on the body of the article and return all entities
def predict_NER_def(text):
    try:
        if (text == None or text == ""):
            return None
        
        # Extract entities from the text
        entities = []
        for entity in nlp(text).ents:
            entities.append((entity.text, entity.label_))

        if (len(entities) > 0):
            return entities
        else: 
            return None
    except Exception as error:
        print(error) # Double prints error?
        return None

In [32]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return None

In [33]:
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

 55%|█████▌    | 11/20 [31:33<32:10, 214.52s/it]

Has location from title: How new MLB rule could change baseball games this season


100%|██████████| 20/20 [50:57<00:00, 152.85s/it]


Filter based on superb specific places such as 'FAC'.

In [34]:
# Filter out the entities that are not locations and sort them by priorities
def filter_locations(entities):

    if (entities == None):
        return None
    
    locations = []
    for entity in entities:
        # Check if entity is of the format (location, label)
        if (len(entity) >= 2): 
            location, label = entity
            
            # Geopolitical Entity but not general like Boston / MA
            invalid_GPE = ["Boston", "Massachusetts"] # TODO: Add more when expanding to Greater Boston
            
            # TODO: Check that they aren't unwanted and return instead after finding a facility or organization if no facility is found 
            # If the entity is a location, add it to the list
            if (("GPE" in label and location not in invalid_GPE) 
                 or ("ORG" in label) # Organization
                 or ("FAC" in label) # Facility
                 or ("LOC" in label) # Location
                ):
                locations.append((location, label.strip()))
    
    # Sort the locations by priority
    priority_order = {'FAC': 1, 'ORG': 2, 'LOC': 3, 'GPE': 4}
    sorted_locations = sorted(locations, key=lambda entity: priority_order[entity[1]])
    
    return sorted_locations

In [35]:
# TODO: Won't be necessary after making other changes
df['NER_Pass_Sorted'] = df['NER_Pass'].progress_apply(filter_locations)

100%|██████████| 20/20 [00:00<00:00, 20054.05it/s]


In [36]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Pass_Sorted
7311,0000017e-9213-d7bb-a97f-bff3f38a0001,WPI student found dead in apartment marking th...,Another Worcester Polytechnic Institute studen...,None,None,"[(Worcester Polytechnic Institute, ORG), (toda...","[(Worcester Polytechnic Institute, ORG), (WPI,..."
4153,00000179-d37f-d770-a57f-f77f3c9c0001,The Home For Little Wanderers Sees Increased N...,year of shutdowns and remote learning has impa...,None,None,"[(Lesli Suggs, PERSON), (Home for Little Wande...","[(The Home, FAC), (Home for Little Wanderers, ..."
4674,0000017a-8257-dc5c-a77a-8ad76dfe0001,The Ever Given Has Set Sail From The Suez Cana...,ISMAILIA Egypt Egyptian authorities announced ...,None,None,"[(ISMAILIA, GPE), (Egypt, GPE), (Egyptian, NOR...","[(the Suez Canal, FAC), (the Suez Canal, FAC),..."
6419,0000017d-1962-d269-a3fd-bf7b76d40001,Republicans and Democrats contemplate future w...,Let assume you have spent at least few minutes...,None,None,"[(at least few minutes, TIME), (this week, DAT...","[(the White House, FAC), (Capitol, FAC), (GOP,..."
6379,0000017d-0c0e-d7af-a3ff-ec3e1c440001,For us Veterans Day is every day says one vet ...,Carlos De Le who grew up in Lawrence served in...,None,None,"[(Carlos De Le, PERSON), (Lawrence, GPE), (Arm...","[(Army, ORG), (Congress, ORG), (the U.S. Depar..."
10694,00000183-f771-d28d-a9f3-f7f9d78f0001,Sweeps of Mass. and Cass encampments return as...,Following city sanctioned sweep of new encampm...,None,None,"[(Massachusetts Avenue, FAC), (Melnea Cass Bou...","[(Massachusetts Avenue, FAC), (Melnea Cass Bou..."
10770,00000184-18e3-d508-a3bf-5eff52940003,NPR reporting on Oregon theater death threats ...,When Nataki Garrett began to receive death thr...,None,None,"[(Nataki Garrett, PERSON), (early this year, D...","[(Oregon Shakespeare Festival, ORG), (OSF, ORG..."
6351,0000017d-045b-df97-afff-3f5f040b0001,secret tape made after Columbine shows the NRA...,Soon after the Columbine High School shooting ...,None,None,"[(Columbine High School, ORG), (1999, DATE), (...","[(Columbine, FAC), (Columbine, FAC), (Columbin..."
5433,0000017b-989c-df60-a9ff-f99fd2b20001,How To Avoid The Next Afghanistan Follow The U...,World War II was the last sustained military e...,None,None,"[(World War II, EVENT), (the United States, GP...","[(the Bay of Pigs, FAC), (University Hall, FAC..."
10484,00000183-a2e7-d687-afef-feff35e80001,What at stake for the Voting Rights Act in the...,It early October which means new term is offic...,None,None,"[(the U.S. Supreme Court, ORG), (Massachusetts...","[(the U.S. Supreme Court, ORG), (Massachusetts..."


This is only necessary in this version. Can delete on alternative proposed method.

In [49]:
unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [55]:
# Return the first most specific location that is not in the unwanted list
def getSpecificLocation(entities, valid_labels):
    if (entities == None or len(entities) == 0):
        return None
    
    # TODO: Do this in the filter_locations function
    # Look through sorted entities, return first entity that is not unwanted
    for (location, label) in entities:
        if label in valid_labels:
            if location not in unwanted_entities[label]:
                return location
        else: 
            break
        
    # If no wanted entities found, return the first entity if it's valid
    location, label = entities[0]
    if label in valid_labels:
        return location
    else:
        return None

In [57]:
df['NER_Location'] = df['NER_Pass_Sorted'].progress_apply(lambda article: getSpecificLocation(article, ['FAC']))

100%|██████████| 20/20 [00:00<?, ?it/s]


### Llama Prediction

In [61]:
#TODO: Comply with token limit of 2048 for Llama
# Run the LLM model on the articles that haven't been tagged with a location yet
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Location'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            return run_llm(article['hl1'], article['body'])
    except Exception as error:
        print(error)
        return None

In [62]:
df['llama_prediction'] = df.progress_apply(predict_llama, axis=1)

  0%|          | 0/20 [00:00<?, ?it/s]

  Based on the information provided in the article, I would guess that the location being referred to is Worcester, Massachusetts, specifically the area surrounding Worcester Polytechnic Institute (WPI). Here's my reasoning:
1. Y - The article explicitly mentions Worcester and WPI multiple times throughout the text, indicating that the location being discussed is within or near Worcester.
2. Specific location within Worcester: Based on the information provided in the article, it appears that the student was found dead in an off-campus apartment, which suggests that the location is likely within the surrounding area of WPI, rather than on campus itself.
3. Involved specific locations or organizations: The article mentions WPI, Worcester Polytechnic Institute, and the National Suicide Prevention Lifeline (800-273-8255). These are the only specific locations or organizations mentioned in the article that could influence my decision.
Based on these factors, I would guess that the location 


llama_print_timings:        load time =   41464.75 ms
llama_print_timings:      sample time =      80.99 ms /   241 runs   (    0.34 ms per token,  2975.71 tokens per second)
llama_print_timings: prompt eval time =  161737.92 ms /  1084 tokens (  149.20 ms per token,     6.70 tokens per second)
llama_print_timings:        eval time =   50853.21 ms /   240 runs   (  211.89 ms per token,     4.72 tokens per second)
llama_print_timings:       total time =   92970.25 ms /  1324 tokens
 10%|█         | 2/20 [01:33<13:57, 46.51s/it]Llama.generate: prefix-match hit


Has location from title or NER: The Home For Little Wanderers Sees Increased Need For Foster Homes In Mass.
Has location from title or NER: The Ever Given Has Set Sail From The Suez Canal Months After It Blocked The Waterway
Has location from title or NER: Republicans and Democrats contemplate future without Donald Trump
  Based on the article, I would guess that the location being described is Lawrence, Massachusetts. The article mentions specific locations within Lawrence, such as downtown Boston and Newton, which suggest that the article is focusing on events and ceremonies taking place in that area. Additionally, the article quotes a veteran named Carlos De Le who grew up in Lawrence and served in the Army for 10 years, including two deployments to Iraq. This suggests that the article is specifically focused on the experiences of veterans in Lawrence.
The specific locations explicitly found within the article that influenced my decision are:
* Downtown Boston
* Lawrence, Massachuse


llama_print_timings:        load time =   41464.75 ms
llama_print_timings:      sample time =      90.18 ms /   256 runs   (    0.35 ms per token,  2838.89 tokens per second)
llama_print_timings: prompt eval time =   99413.78 ms /  1019 tokens (   97.56 ms per token,    10.25 tokens per second)
llama_print_timings:        eval time =   53174.33 ms /   255 runs   (  208.53 ms per token,     4.80 tokens per second)
llama_print_timings:       total time =  153334.40 ms /  1274 tokens
 30%|███       | 6/20 [04:06<09:26, 40.46s/it]

...

Llama.generate: prefix-match hit


  Here is my response:
1. Y - I believe the article is talking about a specific region of Boston, specifically the Milton area.
2. The specific location within the city is Milton, Massachusetts.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* The Kelly Clarkson Show (a national television show)
The article mentions that the rock band, The Lazy Susans, was featured on The Kelly Clarkson Show, which suggests that the band is from Boston and has gained national recognition. The article also highlights the fact that the band members are all mothers who started the band during the pandemic, which adds a personal touch to their story and makes it more relatable to readers in the Milton area.


llama_print_timings:        load time =   41464.75 ms
llama_print_timings:      sample time =      58.22 ms /   163 runs   (    0.36 ms per token,  2799.97 tokens per second)
llama_print_timings: prompt eval time =   33974.41 ms /   367 tokens (   92.57 ms per token,    10.80 tokens per second)
llama_print_timings:        eval time =   34125.45 ms /   162 runs   (  210.65 ms per token,     4.75 tokens per second)
llama_print_timings:       total time =   68559.80 ms /   529 tokens
100%|██████████| 20/20 [17:48<00:00, 54.76s/it]Llama.generate: prefix-match hit


  Based on the information provided in the article, I would guess that the location being referred to is Boston, Massachusetts, USA. Here are my reasons for this conclusion:
1. Y - The article does not mention any specific region of Boston, indicating that the location being referred to is likely a broader area within the city.
2. The specific location within Boston that I believe is being referred to is the city's medical community, particularly hospitals and research institutions. The article mentions that officials in both the U.S. and U.K. are excited about the potential of effective COVID-19 pills, which suggests that these locations are involved in the clinical trials and approval process.
3. The article explicitly mentions the following specific locations or organizations that influenced my decision:
* Pfizer's headquarters in New York City, where the company developed its COVID-19 pill.
* Merck's research facility in Kenilworth, New Jersey, where the company developed its COVID


llama_print_timings:        load time =   41464.75 ms
llama_print_timings:      sample time =      90.72 ms /   256 runs   (    0.35 ms per token,  2821.90 tokens per second)
llama_print_timings: prompt eval time =   55454.14 ms /   582 tokens (   95.28 ms per token,    10.50 tokens per second)
llama_print_timings:        eval time =   56268.16 ms /   255 runs   (  220.66 ms per token,     4.53 tokens per second)
llama_print_timings:       total time =  112427.26 ms /   837 tokens
100%|██████████| 20/20 [19:40<00:00, 59.04s/it]


In [63]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Pass_Sorted,NER_Prediction,NER_Pred_Sorted,NER_Location
7311,0000017e-9213-d7bb-a97f-bff3f38a0001,WPI student found dead in apartment marking th...,Another Worcester Polytechnic Institute studen...,Based on the information provided in the art...,None,"[(Worcester Polytechnic Institute, ORG), (toda...","[(Worcester Polytechnic Institute, ORG), (WPI,...",None,None,None
4153,00000179-d37f-d770-a57f-f77f3c9c0001,The Home For Little Wanderers Sees Increased N...,year of shutdowns and remote learning has impa...,None,None,"[(Lesli Suggs, PERSON), (Home for Little Wande...","[(The Home, FAC), (Home for Little Wanderers, ...",None,None,The Home
4674,0000017a-8257-dc5c-a77a-8ad76dfe0001,The Ever Given Has Set Sail From The Suez Cana...,ISMAILIA Egypt Egyptian authorities announced ...,None,None,"[(ISMAILIA, GPE), (Egypt, GPE), (Egyptian, NOR...","[(the Suez Canal, FAC), (the Suez Canal, FAC),...",None,None,the Suez Canal
6419,0000017d-1962-d269-a3fd-bf7b76d40001,Republicans and Democrats contemplate future w...,Let assume you have spent at least few minutes...,None,None,"[(at least few minutes, TIME), (this week, DAT...","[(the White House, FAC), (Capitol, FAC), (GOP,...",None,None,the White House
6379,0000017d-0c0e-d7af-a3ff-ec3e1c440001,For us Veterans Day is every day says one vet ...,Carlos De Le who grew up in Lawrence served in...,"Based on the article, I would guess that the...",None,"[(Carlos De Le, PERSON), (Lawrence, GPE), (Arm...","[(Army, ORG), (Congress, ORG), (the U.S. Depar...",None,None,None
10694,00000183-f771-d28d-a9f3-f7f9d78f0001,Sweeps of Mass. and Cass encampments return as...,Following city sanctioned sweep of new encampm...,None,None,"[(Massachusetts Avenue, FAC), (Melnea Cass Bou...","[(Massachusetts Avenue, FAC), (Melnea Cass Bou...",None,None,Massachusetts Avenue
10770,00000184-18e3-d508-a3bf-5eff52940003,NPR reporting on Oregon theater death threats ...,When Nataki Garrett began to receive death thr...,Based on the information provided in the art...,None,"[(Nataki Garrett, PERSON), (early this year, D...","[(Oregon Shakespeare Festival, ORG), (OSF, ORG...",None,None,None
6351,0000017d-045b-df97-afff-3f5f040b0001,secret tape made after Columbine shows the NRA...,Soon after the Columbine High School shooting ...,None,None,"[(Columbine High School, ORG), (1999, DATE), (...","[(Columbine, FAC), (Columbine, FAC), (Columbin...",None,None,Columbine
5433,0000017b-989c-df60-a9ff-f99fd2b20001,How To Avoid The Next Afghanistan Follow The U...,World War II was the last sustained military e...,None,None,"[(World War II, EVENT), (the United States, GP...","[(the Bay of Pigs, FAC), (University Hall, FAC...",None,None,the Bay of Pigs
10484,00000183-a2e7-d687-afef-feff35e80001,What at stake for the Voting Rights Act in the...,It early October which means new term is offic...,Based on the information provided in the art...,None,"[(the U.S. Supreme Court, ORG), (Massachusetts...","[(the U.S. Supreme Court, ORG), (Massachusetts...",None,None,None


We then apply NER on the llama outputs

In [64]:
df['NER_Prediction'] = df['llama_prediction'].progress_apply(predict_NER_def)

100%|██████████| 20/20 [05:56<00:00, 17.84s/it]


In [103]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Pass_Sorted,NER_Prediction,NER_Pred_Sorted,NER_Location,Locations,Coordinates
7311,0000017e-9213-d7bb-a97f-bff3f38a0001,WPI student found dead in apartment marking th...,Another Worcester Polytechnic Institute studen...,Based on the information provided in the art...,None,"[(Worcester Polytechnic Institute, ORG), (toda...","[(Worcester Polytechnic Institute, ORG), (WPI,...","[(Worcester, GPE), (Massachusetts, GPE), (Worc...","[(Worcester Polytechnic Institute, ORG), (WPI,...",None,Worcester Polytechnic Institute,"[-71.8068416, 42.2746179]"
4153,00000179-d37f-d770-a57f-f77f3c9c0001,The Home For Little Wanderers Sees Increased N...,year of shutdowns and remote learning has impa...,None,None,"[(Lesli Suggs, PERSON), (Home for Little Wande...","[(The Home, FAC), (Home for Little Wanderers, ...",None,None,The Home,The Home,"[-71.3824374, 42.4072107]"
4674,0000017a-8257-dc5c-a77a-8ad76dfe0001,The Ever Given Has Set Sail From The Suez Cana...,ISMAILIA Egypt Egyptian authorities announced ...,None,None,"[(ISMAILIA, GPE), (Egypt, GPE), (Egyptian, NOR...","[(the Suez Canal, FAC), (the Suez Canal, FAC),...",None,None,the Suez Canal,the Suez Canal,"[-71.3824374, 42.4072107]"
6419,0000017d-1962-d269-a3fd-bf7b76d40001,Republicans and Democrats contemplate future w...,Let assume you have spent at least few minutes...,None,None,"[(at least few minutes, TIME), (this week, DAT...","[(the White House, FAC), (Capitol, FAC), (GOP,...",None,None,the White House,the White House,"[-71.3824374, 42.4072107]"
6379,0000017d-0c0e-d7af-a3ff-ec3e1c440001,For us Veterans Day is every day says one vet ...,Carlos De Le who grew up in Lawrence served in...,"Based on the article, I would guess that the...",None,"[(Carlos De Le, PERSON), (Lawrence, GPE), (Arm...","[(Army, ORG), (Congress, ORG), (the U.S. Depar...","[(Lawrence, GPE), (Massachusetts, GPE), (Lawre...","[(Army, ORG), (Congress, ORG), (4th Congressio...",None,Army,"[-71.3824374, 42.4072107]"
10694,00000183-f771-d28d-a9f3-f7f9d78f0001,Sweeps of Mass. and Cass encampments return as...,Following city sanctioned sweep of new encampm...,None,None,"[(Massachusetts Avenue, FAC), (Melnea Cass Bou...","[(Massachusetts Avenue, FAC), (Melnea Cass Bou...",None,None,Massachusetts Avenue,Massachusetts Avenue,"[-71.1370282, 42.4014632]"
10770,00000184-18e3-d508-a3bf-5eff52940003,NPR reporting on Oregon theater death threats ...,When Nataki Garrett began to receive death thr...,Based on the information provided in the art...,None,"[(Nataki Garrett, PERSON), (early this year, D...","[(Oregon Shakespeare Festival, ORG), (OSF, ORG...","[(Ashland, GPE), (Oregon, GPE), (the Oregon Sh...","[(the Oregon Shakespeare Festival, ORG), (OSF,...",None,the Oregon Shakespeare Festival,"[-120.5542012, 43.8041334]"
6351,0000017d-045b-df97-afff-3f5f040b0001,secret tape made after Columbine shows the NRA...,Soon after the Columbine High School shooting ...,None,None,"[(Columbine High School, ORG), (1999, DATE), (...","[(Columbine, FAC), (Columbine, FAC), (Columbin...",None,None,Columbine,Columbine,"[-71.3824374, 42.4072107]"
5433,0000017b-989c-df60-a9ff-f99fd2b20001,How To Avoid The Next Afghanistan Follow The U...,World War II was the last sustained military e...,None,None,"[(World War II, EVENT), (the United States, GP...","[(the Bay of Pigs, FAC), (University Hall, FAC...",None,None,the Bay of Pigs,the Bay of Pigs,"[-71.3824374, 42.4072107]"
10484,00000183-a2e7-d687-afef-feff35e80001,What at stake for the Voting Rights Act in the...,It early October which means new term is offic...,Based on the information provided in the art...,None,"[(the U.S. Supreme Court, ORG), (Massachusetts...","[(the U.S. Supreme Court, ORG), (Massachusetts...","[(Boston, GPE), (Massachusetts, GPE), (the Mas...","[(the Massachusetts Supreme Judicial Court, OR...",None,the Massachusetts Supreme Judicial Court,"[-71.3824374, 42.4072107]"


Sort the NER Predictions

In [104]:
df['NER_Pred_Sorted'] = df['NER_Prediction'].progress_apply(filter_locations)

100%|██████████| 20/20 [00:00<?, ?it/s]


## Collect Most Specific Locations

In [105]:
# TODO: Populate the unwanted entities cache
unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [106]:
## TODO: DELETE AFTER POPULATING THE UNWANTED ENTITIES CACHE
# unwanted_entities = {
#     'FAC': ['Boston'],
#     'ORG': ['New York Times'],
#     'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
#     'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
# }

# save_cache_to_file(unwanted_entities, unwanted_entities_path)

Filter out locations if they are unwanted and by priority

In [107]:
# Return the first most specific location that is not in the unwanted list
def getSpecificLocation(entities, valid_labels):
    if (entities == None or len(entities) == 0):
        return None
    
    # TODO: Do this in the filter_locations function
    # Look through sorted entities, return first entity that is not unwanted
    for (location, label) in entities:
        if label in valid_labels:
            if location not in unwanted_entities[label]:
                return location
        else: 
            break
        
    # If no wanted entities found, return the first entity if it's valid
    location, label = entities[0]
    if label in valid_labels:
        return location
    else:
        return None

Extract locations from the most specific pass

In [108]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    # Get Location from Explicit Pass if available
    entity = article['Explicit_Pass']
    if (entity != None): 
        return entity[0]
    
    # Get Location from NER Pass if available
    entity = getSpecificLocation(article['NER_Pass_Sorted'], valid_labels=['FAC'])
    if (entity != None): 
        return entity
    
    # Get Location from NER Prediction if available
    entity = getSpecificLocation(article['NER_Pred_Sorted'], valid_labels=['FAC', 'ORG'])
    if (entity != None): 
        return entity
    
    return None # Must be a very hard/bad article :-(

In [109]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

100%|██████████| 20/20 [00:00<00:00, 1278.46it/s]


In [110]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Pass_Sorted,NER_Prediction,NER_Pred_Sorted,NER_Location,Locations,Coordinates
7311,0000017e-9213-d7bb-a97f-bff3f38a0001,WPI student found dead in apartment marking th...,Another Worcester Polytechnic Institute studen...,Based on the information provided in the art...,None,"[(Worcester Polytechnic Institute, ORG), (toda...","[(Worcester Polytechnic Institute, ORG), (WPI,...","[(Worcester, GPE), (Massachusetts, GPE), (Worc...","[(Worcester Polytechnic Institute, ORG), (WPI,...",None,Worcester Polytechnic Institute,"[-71.8068416, 42.2746179]"
4153,00000179-d37f-d770-a57f-f77f3c9c0001,The Home For Little Wanderers Sees Increased N...,year of shutdowns and remote learning has impa...,None,None,"[(Lesli Suggs, PERSON), (Home for Little Wande...","[(The Home, FAC), (Home for Little Wanderers, ...",None,None,The Home,The Home,"[-71.3824374, 42.4072107]"
4674,0000017a-8257-dc5c-a77a-8ad76dfe0001,The Ever Given Has Set Sail From The Suez Cana...,ISMAILIA Egypt Egyptian authorities announced ...,None,None,"[(ISMAILIA, GPE), (Egypt, GPE), (Egyptian, NOR...","[(the Suez Canal, FAC), (the Suez Canal, FAC),...",None,None,the Suez Canal,the Suez Canal,"[-71.3824374, 42.4072107]"
6419,0000017d-1962-d269-a3fd-bf7b76d40001,Republicans and Democrats contemplate future w...,Let assume you have spent at least few minutes...,None,None,"[(at least few minutes, TIME), (this week, DAT...","[(the White House, FAC), (Capitol, FAC), (GOP,...",None,None,the White House,the White House,"[-71.3824374, 42.4072107]"
6379,0000017d-0c0e-d7af-a3ff-ec3e1c440001,For us Veterans Day is every day says one vet ...,Carlos De Le who grew up in Lawrence served in...,"Based on the article, I would guess that the...",None,"[(Carlos De Le, PERSON), (Lawrence, GPE), (Arm...","[(Army, ORG), (Congress, ORG), (the U.S. Depar...","[(Lawrence, GPE), (Massachusetts, GPE), (Lawre...","[(Army, ORG), (Congress, ORG), (4th Congressio...",None,Army,"[-71.3824374, 42.4072107]"
10694,00000183-f771-d28d-a9f3-f7f9d78f0001,Sweeps of Mass. and Cass encampments return as...,Following city sanctioned sweep of new encampm...,None,None,"[(Massachusetts Avenue, FAC), (Melnea Cass Bou...","[(Massachusetts Avenue, FAC), (Melnea Cass Bou...",None,None,Massachusetts Avenue,Massachusetts Avenue,"[-71.1370282, 42.4014632]"
10770,00000184-18e3-d508-a3bf-5eff52940003,NPR reporting on Oregon theater death threats ...,When Nataki Garrett began to receive death thr...,Based on the information provided in the art...,None,"[(Nataki Garrett, PERSON), (early this year, D...","[(Oregon Shakespeare Festival, ORG), (OSF, ORG...","[(Ashland, GPE), (Oregon, GPE), (the Oregon Sh...","[(the Oregon Shakespeare Festival, ORG), (OSF,...",None,the Oregon Shakespeare Festival,"[-120.5542012, 43.8041334]"
6351,0000017d-045b-df97-afff-3f5f040b0001,secret tape made after Columbine shows the NRA...,Soon after the Columbine High School shooting ...,None,None,"[(Columbine High School, ORG), (1999, DATE), (...","[(Columbine, FAC), (Columbine, FAC), (Columbin...",None,None,Columbine,Columbine,"[-71.3824374, 42.4072107]"
5433,0000017b-989c-df60-a9ff-f99fd2b20001,How To Avoid The Next Afghanistan Follow The U...,World War II was the last sustained military e...,None,None,"[(World War II, EVENT), (the United States, GP...","[(the Bay of Pigs, FAC), (University Hall, FAC...",None,None,the Bay of Pigs,the Bay of Pigs,"[-71.3824374, 42.4072107]"
10484,00000183-a2e7-d687-afef-feff35e80001,What at stake for the Voting Rights Act in the...,It early October which means new term is offic...,Based on the information provided in the art...,None,"[(the U.S. Supreme Court, ORG), (Massachusetts...","[(the U.S. Supreme Court, ORG), (Massachusetts...","[(Boston, GPE), (Massachusetts, GPE), (the Mas...","[(the Massachusetts Supreme Judicial Court, OR...",None,the Massachusetts Supreme Judicial Court,"[-71.3824374, 42.4072107]"


## Get the coordinates

In [111]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [112]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [113]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

100%|██████████| 20/20 [00:00<?, ?it/s]


In [114]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Pass_Sorted,NER_Prediction,NER_Pred_Sorted,NER_Location,Locations,Coordinates
7311,0000017e-9213-d7bb-a97f-bff3f38a0001,WPI student found dead in apartment marking th...,Another Worcester Polytechnic Institute studen...,Based on the information provided in the art...,None,"[(Worcester Polytechnic Institute, ORG), (toda...","[(Worcester Polytechnic Institute, ORG), (WPI,...","[(Worcester, GPE), (Massachusetts, GPE), (Worc...","[(Worcester Polytechnic Institute, ORG), (WPI,...",None,Worcester Polytechnic Institute,"[-71.8068416, 42.2746179]"
4153,00000179-d37f-d770-a57f-f77f3c9c0001,The Home For Little Wanderers Sees Increased N...,year of shutdowns and remote learning has impa...,None,None,"[(Lesli Suggs, PERSON), (Home for Little Wande...","[(The Home, FAC), (Home for Little Wanderers, ...",None,None,The Home,The Home,"[-71.3824374, 42.4072107]"
4674,0000017a-8257-dc5c-a77a-8ad76dfe0001,The Ever Given Has Set Sail From The Suez Cana...,ISMAILIA Egypt Egyptian authorities announced ...,None,None,"[(ISMAILIA, GPE), (Egypt, GPE), (Egyptian, NOR...","[(the Suez Canal, FAC), (the Suez Canal, FAC),...",None,None,the Suez Canal,the Suez Canal,"[-71.3824374, 42.4072107]"
6419,0000017d-1962-d269-a3fd-bf7b76d40001,Republicans and Democrats contemplate future w...,Let assume you have spent at least few minutes...,None,None,"[(at least few minutes, TIME), (this week, DAT...","[(the White House, FAC), (Capitol, FAC), (GOP,...",None,None,the White House,the White House,"[-71.3824374, 42.4072107]"
6379,0000017d-0c0e-d7af-a3ff-ec3e1c440001,For us Veterans Day is every day says one vet ...,Carlos De Le who grew up in Lawrence served in...,"Based on the article, I would guess that the...",None,"[(Carlos De Le, PERSON), (Lawrence, GPE), (Arm...","[(Army, ORG), (Congress, ORG), (the U.S. Depar...","[(Lawrence, GPE), (Massachusetts, GPE), (Lawre...","[(Army, ORG), (Congress, ORG), (4th Congressio...",None,Army,"[-71.3824374, 42.4072107]"
10694,00000183-f771-d28d-a9f3-f7f9d78f0001,Sweeps of Mass. and Cass encampments return as...,Following city sanctioned sweep of new encampm...,None,None,"[(Massachusetts Avenue, FAC), (Melnea Cass Bou...","[(Massachusetts Avenue, FAC), (Melnea Cass Bou...",None,None,Massachusetts Avenue,Massachusetts Avenue,"[-71.1370282, 42.4014632]"
10770,00000184-18e3-d508-a3bf-5eff52940003,NPR reporting on Oregon theater death threats ...,When Nataki Garrett began to receive death thr...,Based on the information provided in the art...,None,"[(Nataki Garrett, PERSON), (early this year, D...","[(Oregon Shakespeare Festival, ORG), (OSF, ORG...","[(Ashland, GPE), (Oregon, GPE), (the Oregon Sh...","[(the Oregon Shakespeare Festival, ORG), (OSF,...",None,the Oregon Shakespeare Festival,"[-120.5542012, 43.8041334]"
6351,0000017d-045b-df97-afff-3f5f040b0001,secret tape made after Columbine shows the NRA...,Soon after the Columbine High School shooting ...,None,None,"[(Columbine High School, ORG), (1999, DATE), (...","[(Columbine, FAC), (Columbine, FAC), (Columbin...",None,None,Columbine,Columbine,"[-71.3824374, 42.4072107]"
5433,0000017b-989c-df60-a9ff-f99fd2b20001,How To Avoid The Next Afghanistan Follow The U...,World War II was the last sustained military e...,None,None,"[(World War II, EVENT), (the United States, GP...","[(the Bay of Pigs, FAC), (University Hall, FAC...",None,None,the Bay of Pigs,the Bay of Pigs,"[-71.3824374, 42.4072107]"
10484,00000183-a2e7-d687-afef-feff35e80001,What at stake for the Voting Rights Act in the...,It early October which means new term is offic...,Based on the information provided in the art...,None,"[(the U.S. Supreme Court, ORG), (Massachusetts...","[(the U.S. Supreme Court, ORG), (Massachusetts...","[(Boston, GPE), (Massachusetts, GPE), (the Mas...","[(the Massachusetts Supreme Judicial Court, OR...",None,the Massachusetts Supreme Judicial Court,"[-71.3824374, 42.4072107]"


## Geocode locations

In [115]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [122]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract =  known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        # coordinates = known_locations[location]["coordinates"]
        coordinates = [10, 10]
        print(coordinates)
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [ ]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)


In [ ]:
df.head(10)

In [ ]:
len(df)

In [ ]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [ ]:
print(len(df))
df.head(10)

## Topic Modeling

In [ ]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [ ]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns